# LeetCode #1104 — Path In Zigzag Labelled Binary Tree

**Bài Medium · Python 3 · Cây nhị phân / Toán học / Đánh số theo tầng**  
Đề chính thức: https://leetcode.com/problems/path-in-zigzag-labelled-binary-tree/

```python
class Solution:
    def pathInZigZagTree(self, label: int) -> List[int]:
        ...
```

Notebook này dành cho người mới. Bạn sẽ tự suy ra công thức **tìm cha** trong cây đánh số zigzag; không cần tạo cây khổng lồ.

**Cách học:** đọc markdown → tự đoán kết quả → chạy cell code ngay dưới → so sánh với cách giải thích.

## 1. Mục tiêu học

Sau khi học, bạn có thể:
1. Hiểu *binary tree* và cách đánh số theo từng tầng (*level*).
2. Tìm khoảng nhãn `[low, high]` của tầng chứa một nhãn bất kỳ.
3. Hiểu phép **phản chiếu** (mirror) nhãn trong một tầng và suy ra công thức cha.
4. Viết lời giải `O(log label)` với `List[int]` đúng chuẩn LeetCode.
5. Trace các bước, xử lý biên và tự giải 5 bài thực hành.

## 2. Đọc đề và xem cây bằng mắt

Cây nhị phân đầy đủ, mỗi nút có **hai con**. Đánh số theo từng tầng:
- Tầng **lẻ** (1, 3, 5, ...): nhãn từ trái sang phải.
- Tầng **chẵn** (2, 4, 6, ...): nhãn từ phải sang trái.

```text
Tầng 1:                      1
Tầng 2:                3           2
Tầng 3:             4     5     6     7
Tầng 4:           15 14 13 12 11 10  9  8
```

Cho `label` của nút đích, **trả về đường đi từ gốc đến nút đó** (gồm cả hai đầu).

- Ví dụ 1: `label = 14` → `[1, 3, 4, 14]`.
- Ví dụ 2: `label = 26` → `[1, 2, 6, 10, 26]`.
- Ràng buộc: `1 <= label <= 10**6`.

> Đây là **nhãn thực trong cây zigzag**, không phải chỉ số của một mảng Python.

## 3. Kiến thức nền — cây nhị phân và thứ tự theo tầng

- **Nút gốc** (*root*): nút `1`, không có cha.
- **Nút cha** (*parent*): nút đứng ngay phía trên, nối trực tiếp với nút đang xét.
- **Tầng** (*level*): gốc ở tầng 1, các con ở tầng 2, ...
- Mỗi tầng có số nút **gấp đôi** tầng trước.

Khi đánh số bình thường từ trái sang phải trên *mọi* tầng, các dải nhãn là `1`, `2..3`, `4..7`, `8..15`, ...

In [1]:
def normal_rows(number_of_levels):
    rows = []
    for level in range(1, number_of_levels + 1):
        low = 2 ** (level - 1)
        high = 2 ** level - 1
        rows.append(list(range(low, high + 1)))
    return rows

for level, row in enumerate(normal_rows(4), start=1):
    print(f'Tầng {level}: {row}')

Tầng 1: [1]
Tầng 2: [2, 3]
Tầng 3: [4, 5, 6, 7]
Tầng 4: [8, 9, 10, 11, 12, 13, 14, 15]


### 3.1. Khác biệt nằm ở **hướng nhãn**, không phải cấu trúc cây

Trên tầng chẵn, ta đảo thứ tự **nhãn được viết từ trái sang phải**. Mỗi vị trí trong cây vẫn nối tới cha có cùng vị trí như cây bình thường.

Mẹo nhớ: **đổi nhãn, không đổi nhánh cây**.

In [2]:
def zigzag_rows(number_of_levels):
    rows = []
    for level, row in enumerate(normal_rows(number_of_levels), start=1):
        rows.append(row if level % 2 == 1 else row[::-1])
    return rows

for level, row in enumerate(zigzag_rows(5), start=1):
    print(f'Tầng {level}: {row}')

Tầng 1: [1]
Tầng 2: [3, 2]
Tầng 3: [4, 5, 6, 7]
Tầng 4: [15, 14, 13, 12, 11, 10, 9, 8]
Tầng 5: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


## 4. Python nền tảng — lũy thừa, chia nguyên, đảo danh sách

Ta sẽ gặp ba cú pháp:
- `2 ** k`: lũy thừa, ví dụ `2 ** 3 = 8`.
- `a // 2`: chia lấy phần nguyên, ví dụ `15 // 2 = 7`.
- `a[::-1]`: tạo danh sách có thứ tự ngược lại; `a.reverse()` thì sửa ngay danh sách.

Sau khi đi từ **nút đích ngược lên gốc**, ta cần đảo kết quả để được gốc → đích.

In [3]:
print('2**3 =', 2 ** 3)
print('15//2 =', 15 // 2)
sample = [14, 4, 3, 1]
print('Từ đích về gốc:', sample)
print('Từ gốc tới đích:', sample[::-1])

2**3 = 8
15//2 = 7
Từ đích về gốc: [14, 4, 3, 1]
Từ gốc tới đích: [1, 3, 4, 14]


### 4.1. `bit_length()` có ích gì?

`x.bit_length()` là số bit cần để viết số nguyên dương `x` trong hệ nhị phân.

Ví dụ: `14 = 1110₂` có 4 bit → nằm ở **tầng 4**. Tổng quát với `x >= 1`:

`level = x.bit_length()` và `2**(level-1) <= x <= 2**level - 1`.

Không cần biết hệ nhị phân trước: có thể hiểu `bit_length` là cách ngắn để tìm tầng chứa `x`.

In [4]:
for x in [1, 2, 3, 4, 7, 8, 14, 26]:
    level = x.bit_length()
    low, high = 2 ** (level - 1), 2 ** level - 1
    print(f'x={x:2d}  bit={x:>5b}  tầng={level}  dải=[{low},{high}]')

x= 1  bit=    1  tầng=1  dải=[1,1]
x= 2  bit=   10  tầng=2  dải=[2,3]
x= 3  bit=   11  tầng=2  dải=[2,3]
x= 4  bit=  100  tầng=3  dải=[4,7]
x= 7  bit=  111  tầng=3  dải=[4,7]
x= 8  bit= 1000  tầng=4  dải=[8,15]
x=14  bit= 1110  tầng=4  dải=[8,15]
x=26  bit=11010  tầng=5  dải=[16,31]


## 5. Cách trực giác — xây các tầng và tra cha

Ta có thể dựng đủ các tầng từ `1` đến tầng chứa `label`.

Nếu nút ở **vị trí `j`** của tầng hiện tại (tính từ 0), cha nó ở **vị trí `j // 2`** của tầng ngay phía trên. Điều này đúng vì mỗi cha có hai con liên tiếp.

Cách này dễ kiểm tra cho nhãn nhỏ nhưng **không phù hợp `label = 10**6`** vì phải tạo rất nhiều nút.

In [5]:
def path_by_building_rows(label):
    rows = zigzag_rows(label.bit_length())
    index = rows[-1].index(label)
    result = []
    for level in range(len(rows) - 1, -1, -1):
        result.append(rows[level][index])
        index //= 2
    return result[::-1]

for label in [1, 3, 4, 14, 26]:
    print(label, '→', path_by_building_rows(label))

1 → [1]
3 → [1, 3]
4 → [1, 3, 4]
14 → [1, 3, 4, 14]
26 → [1, 2, 6, 10, 26]


### 5.1. Điểm yếu của cách dựng cây

Với nhãn gần `10**6`, ta phải giữ khoảng một triệu nhãn chỉ để tìm đường đi **dài chưa đến 21 nút**. Đây là dấu hiệu cần một cách tính **cha trực tiếp từ nhãn**.

**Câu hỏi đột phá:** biết nhãn `x` ở tầng `k`, có thể tính nhãn của cha mà không vẽ cây không?

## 6. Bước đột phá — phép phản chiếu trong một tầng

Giả sử tầng có nhãn từ `low` đến `high`. Hai nhãn ở vị trí đối xứng nhau qua tâm luôn có tổng bằng `low + high`.

Do đó, **nhãn đối xứng** của `x` là:

`mirror(x) = low + high - x`.

Ví dụ tầng 4 có nhãn `[8..15]`: `mirror(14) = 8 + 15 - 14 = 9`.

Trong cây đánh số bình thường, nhãn `9` có cha `9 // 2 = 4`. Trong cây zigzag, **nút mang nhãn 14** chính là nút ở vị trí của `9` trong cách đánh số thường.

In [6]:
low, high = 8, 15
for x in [8, 9, 10, 13, 14, 15]:
    print(f'{x:2d} đối xứng với {low + high - x:2d}')

 8 đối xứng với 15
 9 đối xứng với 14
10 đối xứng với 13
13 đối xứng với 10
14 đối xứng với  9
15 đối xứng với  8


### 6.1. Công thức tìm cha thực sự

Từ `x` đang nằm trong tầng có khoảng `[low, high]`, ta có:

`parent(x) = (low + high - x) // 2`.

**Vì sao không cần phản chiếu lần thứ hai?** Hãy coi nhãn `x` ở tầng hiện tại như số đặt trên vị trí của cây đánh số thường: đổi `x` thành số bình thường bằng `mirror`, chia đôi để lên cha. Tầng cha có hướng đánh số ngược với tầng con nên kết quả chia đôi chính là nhãn zigzag của cha.

Xem `14 → (8+15−14)//2 = 4`, rồi `4 → (4+7−4)//2 = 3`, rồi `3 → (2+3−3)//2 = 1`.

In [7]:
def parent_zigzag(x):
    if x == 1:
        return None  # Nút gốc không có cha.
    level = x.bit_length()
    low = 1 << (level - 1)
    high = (1 << level) - 1
    return (low + high - x) // 2

for x in [14, 4, 3, 26, 10, 6, 2, 1]:
    print(f'Cha của {x}: {parent_zigzag(x)}')

Cha của 14: 4
Cha của 4: 3
Cha của 3: 1
Cha của 26: 10
Cha của 10: 6
Cha của 6: 2
Cha của 2: 1
Cha của 1: None


### 6.2. Cú pháp `<<` có nghĩa gì?

`1 << k` dịch bit `1` sang trái `k` vị trí và bằng `2 ** k` với số nguyên `k >= 0`.

Trong code nộp, bạn có thể dùng **`2 ** k` thay cho `1 << k`** nếu thấy dễ đọc hơn. Hai cách đều tính cùng `low` và `high`.

In [8]:
for k in range(5):
    print(f'k={k}: 1 << k = {1 << k}, 2**k = {2 ** k}')
    assert 1 << k == 2 ** k

k=0: 1 << k = 1, 2**k = 1
k=1: 1 << k = 2, 2**k = 2
k=2: 1 << k = 4, 2**k = 4
k=3: 1 << k = 8, 2**k = 8
k=4: 1 << k = 16, 2**k = 16


## 7. TRACE bằng tay — `label = 14`

| Nhãn hiện tại `x` | Tầng | `low` | `high` | `mirror = low+high-x` | Cha `mirror//2` |
|---:|---:|---:|---:|---:|---:|
| 14 | 4 | 8 | 15 | 9 | 4 |
| 4 | 3 | 4 | 7 | 7 | 3 |
| 3 | 2 | 2 | 3 | 2 | 1 |
| 1 | 1 | 1 | 1 | 1 | Dừng |

Ta đi ngược được `[14, 4, 3, 1]` → đảo lại `[1, 3, 4, 14]`.

In [9]:
def trace_path_zigzag(label):
    x = label
    reverse_path = []
    while x >= 1:
        k = x.bit_length()
        low, high = 1 << (k - 1), (1 << k) - 1
        reverse_path.append(x)
        if x == 1:
            print(f'x={x}: tới gốc, DỪNG')
            break
        mirror = low + high - x
        parent = mirror // 2
        print(f'x={x:>2}, tầng={k}, [{low},{high}], '
              f'mirror={mirror}, cha={parent}')
        x = parent
    print('Ngược:', reverse_path)
    print('Đáp án:', reverse_path[::-1])
    return reverse_path[::-1]

assert trace_path_zigzag(14) == [1, 3, 4, 14]

x=14, tầng=4, [8,15], mirror=9, cha=4
x= 4, tầng=3, [4,7], mirror=7, cha=3
x= 3, tầng=2, [2,3], mirror=2, cha=1
x=1: tới gốc, DỪNG
Ngược: [14, 4, 3, 1]
Đáp án: [1, 3, 4, 14]


## 8. Viết lời giải tối ưu từ các mảnh ghép

Giữ một biến `x` là nút đang xét; mỗi vòng lặp:
1. Ghi `x` vào `path`.
2. Nếu `x == 1`, dừng.
3. Dùng `bit_length()` tìm tầng → tính `low`, `high`.
4. Nhảy thẳng lên cha qua công thức phản chiếu.

Cuối cùng, đảo `path` vì đã đi từ đích về gốc.

In [10]:
def path_in_zigzag_tree(label: int) -> list[int]:
    path = []
    x = label
    while x > 0:
        path.append(x)
        level = x.bit_length()
        low = 1 << (level - 1)
        high = (1 << level) - 1
        x = (low + high - x) // 2
    path.reverse()
    return path

print(path_in_zigzag_tree(14))
print(path_in_zigzag_tree(26))

[1, 3, 4, 14]
[1, 2, 6, 10, 26]


### 8.1. Vì sao `while x > 0` tự dừng?

Ở tầng 1, `x = 1`: `low=high=1`, nên `x` tiếp theo bằng `(1+1−1)//2 = 0`.

Tầng cao hơn thì giá trị cha thuộc tầng ngay trên và nhỏ hơn `x`. Không có vòng lặp vô hạn.

In [11]:
for value in [1, 2, 3, 4, 14, 26, 10**6]:
    result = path_in_zigzag_tree(value)
    print(f'label={value:<7}  chiều dài={len(result):>2}  đường đi={result}')
    assert result[0] == 1 and result[-1] == value

label=1        chiều dài= 1  đường đi=[1]
label=2        chiều dài= 2  đường đi=[1, 2]
label=3        chiều dài= 2  đường đi=[1, 3]
label=4        chiều dài= 3  đường đi=[1, 3, 4]
label=14       chiều dài= 4  đường đi=[1, 3, 4, 14]
label=26       chiều dài= 5  đường đi=[1, 2, 6, 10, 26]
label=1000000  chiều dài=20  đường đi=[1, 3, 4, 15, 17, 61, 69, 244, 279, 976, 1118, 3906, 4475, 15625, 17901, 62500, 71607, 250000, 286431, 1000000]


## 9. Code nộp LeetCode — đúng tên lớp, tên phương thức

Trên LeetCode, hãy chọn **Python3**, chỉ nộp nội dung sau. `List` được import rõ ràng để code có thể chạy độc lập cả ở Jupyter lẫn LeetCode.

In [12]:
from typing import List

class Solution:
    def pathInZigZagTree(self, label: int) -> List[int]:
        ans = []
        while label > 0:
            ans.append(label)
            level = label.bit_length()
            low = 1 << (level - 1)
            high = (1 << level) - 1
            label = (low + high - label) // 2
        ans.reverse()
        return ans

assert Solution().pathInZigZagTree(14) == [1, 3, 4, 14]
assert Solution().pathInZigZagTree(26) == [1, 2, 6, 10, 26]
print('Đã qua 2 ví dụ chính thức')

Đã qua 2 ví dụ chính thức


## 10. Vì sao thuật toán đúng? (bất biến vòng lặp)

**Bất biến:** trước mỗi vòng lặp, `label` là nhãn của nút đang đứng trên **đường đi duy nhất** từ đích về gốc; `ans` chứa đúng các nút đã đi qua, theo thứ tự từ dưới lên.

- **Bắt đầu:** đứng ở đúng nút đích; `ans` rỗng, nên bất biến đúng.
- **Duy trì:** công thức `(low+high-label)//2` tìm **đúng cha** của nút hiện tại. Ghi nút vào `ans` rồi đi lên cha không làm lạc khỏi đường đi.
- **Dừng:** mỗi bước lên đúng một tầng, cuối cùng qua gốc `1` rồi `label=0`. `ans` chứa đủ đường đi đích → gốc. Đảo lại nhận gốc → đích.

Không nút nào bị bỏ sót hoặc ghi thừa.

## 11. Time / Space Complexity

Đặt `L` là giá trị nhãn đầu vào. Tầng của nút là `d = floor(log2(L))+1`.

| Cách | Thời gian | Bộ nhớ bổ sung |
|---|---|---|
| Dựng cây theo tầng | `O(L)` | `O(L)` |
| Tính cha bằng phản chiếu | `O(log L)` | `O(1)` ngoài kết quả |

Danh sách đáp án chứa `d = O(log L)` nhãn nên **tổng bộ nhớ tính cả kết quả** là `O(log L)`.

## 12. Kiểm thử — các ví dụ, ranh giới tầng, đối chiếu baseline

Một lời giải không chỉ phải chạy đúng `14` và `26`: cần thử cả nhãn `1`, nhãn đầu/cuối mỗi tầng, và các nhãn ngẫu nhiên nhỏ để có thể đối chiếu với cách dựng cây.

In [13]:
cases = {
    1: [1],
    2: [1, 2],
    3: [1, 3],
    4: [1, 3, 4],
    7: [1, 2, 7],
    14: [1, 3, 4, 14],
    26: [1, 2, 6, 10, 26],
}
for label, expected in cases.items():
    actual = Solution().pathInZigZagTree(label)
    assert actual == expected, (label, actual, expected)
print('PASS:', len(cases), 'trường hợp xác định trước')

PASS: 7 trường hợp xác định trước


In [14]:
from random import Random
rng = Random(1104)
test_labels = list(range(1, 150)) + [rng.randint(1, 2**13 - 1) for _ in range(200)]
for x in test_labels:
    assert path_in_zigzag_tree(x) == path_by_building_rows(x), x
print('PASS:', len(test_labels), 'trường hợp đối chiếu với cây dựng thật')

PASS: 349 trường hợp đối chiếu với cây dựng thật


In [15]:
for value in [2**k - 1 for k in range(1, 20)] + [2**k for k in range(0, 20)]:
    res = Solution().pathInZigZagTree(value)
    assert res[-1] == value
    assert len(res) == value.bit_length()
    for child, parent in zip(res[:0:-1], res[-2::-1]):
        assert parent_zigzag(child) == parent
huge_path = Solution().pathInZigZagTree(10**6)
assert huge_path[0] == 1 and huge_path[-1] == 10**6
print('PASS: ranh giới tầng và label=10**6')

PASS: ranh giới tầng và label=10**6


## 13. Lỗi thường gặp (sai → vì sao → sửa)

**Lỗi 1 — dùng `x//2` ngay trên nhãn zigzag.** `14//2=7` nhưng cha thật là `4`. Chỉ số dạng heap chỉ đúng cho cây đánh số **bình thường**.

**Lỗi 2 — quên trừ `x` khi phản chiếu.** Công thức đúng là `low + high - x`.

**Lỗi 3 — tính sai đỉnh tầng:** tầng `k` kết thúc ở `2**k - 1`, không phải `2**k`.

**Lỗi 4 — quên đảo kết quả:** vòng lặp thu được đích → gốc, còn đề muốn gốc → đích.

In [16]:
print('Sai: cha của 14 =', 14 // 2)
print('Đúng: cha của 14 =', parent_zigzag(14))
assert 14 // 2 != parent_zigzag(14)

wrong_order = [14, 4, 3, 1]
print('Cần đảo:', wrong_order[::-1])

Sai: cha của 14 = 7
Đúng: cha của 14 = 4
Cần đảo: [1, 3, 4, 14]


### 13.1. Nhầm hướng đánh số ở tầng 2

Tầng 2 là `[3, 2]`, không phải `[2, 3]` theo thứ tự trái → phải. Do vậy, đường đi tới nhãn `3` là `[1, 3]` nhưng **nhãn nằm bên trái** gốc; còn nhãn `2` ở bên phải.

In [17]:
assert zigzag_rows(2)[1] == [3, 2]
assert path_in_zigzag_tree(3) == [1, 3]
assert path_in_zigzag_tree(2) == [1, 2]
print('PASS: nhận diện hướng tầng 2')

PASS: nhận diện hướng tầng 2


## 14. Nhận dạng pattern & biến thể

Tìm kiếm kỹ thuật này khi đề có:
- Dữ liệu được **đánh số theo tầng**, độ lớn nhãn rất lớn nhưng độ sâu nhỏ.
- Quy tắc **đảo chiều hoặc phản chiếu** theo tầng.
- Cần đường đi **từ một nút về gốc** thay vì thao tác với toàn bộ cây.

**Biến thể:** tính cha zigzag, tính độ sâu, xuất đường đi từ đích về gốc, đếm số nút trên đường đi. Tất cả đều tận dụng `bit_length` và công thức `mirror`.

In [18]:
def depth_of_label(label):
    return label.bit_length()

def ancestors_from_child(label):
    result = []
    while label != 1:
        label = parent_zigzag(label)
        result.append(label)
    return result

assert depth_of_label(26) == 5
assert ancestors_from_child(14) == [4, 3, 1]
print('Độ sâu của 26:', depth_of_label(26))
print('Cha, ông, ... của 14:', ancestors_from_child(14))

Độ sâu của 26: 5
Cha, ông, ... của 14: [4, 3, 1]


## 15. Bài tập tự luyện — đừng xem đáp án trước

**Bài 1 (đoán output):** viết ra `zigzag_rows(3)` và `7.bit_length()`.

**Bài 2 (tính tay):** với `label = 11`, lập bảng từng tầng gồm `x, low, high, mirror, parent` rồi tìm đường đi.

**Bài 3 (TODO):** viết hàm `exercise_parent(label)` trả `None` nếu là gốc, ngược lại trả đúng cha.

**Bài 4 (TODO):** viết lại lời giải `exercise_path(label)` **không gọi** `path_in_zigzag_tree`.

**Bài 5 (biến thể):** viết `exercise_depth(label)` chỉ dùng bit_length, và `exercise_reverse_path(label)` trả đường đi đích → gốc.

In [19]:
def exercise_parent(label):
    # TODO: tìm low, high của tầng chứa label, rồi phản chiếu và chia 2.
    pass

def exercise_path(label):
    # TODO: while, append, đi lên cha; nhớ đảo kết quả.
    pass

def exercise_depth(label):
    # TODO
    pass

def exercise_reverse_path(label):
    # TODO: trả đường đi từ đích về gốc.
    pass

# Mở comment những dòng sau KHI hoàn thành TODO.
# assert exercise_parent(14) == 4
# assert exercise_path(26) == [1,2,6,10,26]
# assert exercise_depth(26) == 5
# assert exercise_reverse_path(14) == [14,4,3,1]

## 16. Gợi ý theo ba mức

| Bài | Mức 1: tư duy | Mức 2: cú pháp | Mức 3: gần code |
|---|---|---|---|
| 1 | Nhớ tầng chẵn đảo nhãn | Dùng `zigzag_rows(3)` | Tầng 3 là `[4,5,6,7]` |
| 2 | Tìm cha ba lần | Tính `low+high-x` | `11 → 6 → 2 → 1` |
| 3 | Cha là nhãn phản chiếu rồi chia đôi | `level = label.bit_length()` | `(2**(level-1) + 2**level - 1 - label)//2` |
| 4 | Đi ngược về gốc | `while label > 0` | `append(label)`, cập nhật cha, `reverse()` |
| 5 | Độ sâu chính là số tầng | `bit_length()` | Đảo thứ tự đi từ gốc bằng `[::-1]` nếu cần |

## 17. Đáp án tham khảo — tự thử xong mới mở

Bài 1: `[[1], [3, 2], [4, 5, 6, 7]]`, và `7.bit_length() = 3`.

Bài 2: `11` ở `[8..15]` → mirror `12`, cha `6` (không phải `5`); `6` ở `[4..7]` → mirror `5`, cha `2`; `2` ở `[2..3]` → mirror `3`, cha `1`. Vậy đường đi đúng là **`[1,2,6,11]`**. Khi tự tính tay, hãy cẩn thận phép phản chiếu!

In [20]:
def answer_parent(label):
    if label == 1:
        return None
    level = label.bit_length()
    low, high = 1 << (level - 1), (1 << level) - 1
    return (low + high - label) // 2

def answer_path(label):
    result = []
    while label:
        result.append(label)
        level = label.bit_length()
        label = ((1 << (level - 1)) + (1 << level) - 1 - label) // 2
    return result[::-1]

def answer_depth(label):
    return label.bit_length()

def answer_reverse_path(label):
    return answer_path(label)[::-1]

assert answer_parent(14) == 4
assert answer_path(11) == [1, 2, 6, 11]
assert answer_path(26) == [1, 2, 6, 10, 26]
assert answer_depth(26) == 5
assert answer_reverse_path(14) == [14, 4, 3, 1]
print('PASS: toàn bộ đáp án tham khảo')

PASS: toàn bộ đáp án tham khảo


## 18. Template nhớ nhanh + checklist nộp bài

```python
level = x.bit_length()
low, high = 1 << (level-1), (1 << level)-1
parent = (low + high - x) // 2
```

- [ ] Tầng 1 chứa `1`; tầng `k` có khoảng `[2^(k-1), 2^k−1]`.
- [ ] Tôi hiểu vì sao phải phản chiếu trước khi tìm cha.
- [ ] Tôi thu thập đường đi từ dưới lên rồi đảo.
- [ ] Tôi đã test `label=1`, `14`, `26`, nhãn ở ranh giới tầng.
- [ ] Complexity: thời gian `O(log label)`; bộ nhớ `O(log label)` tính cả kết quả.

**Câu chốt:** cấu trúc cây không đổi, chỉ có nhãn mỗi tầng bị đảo chiều. Dùng phản chiếu để quay lại phép tìm cha quen thuộc.